# Package Test

Load the exported ONNX package, reuse the saved training artifacts, and compare prediction vs target on the test split.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import tempfile
import zipfile
import sys
sys.path.append("../")

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort

from ibamlkit.training import ConstantFactorTransform, split_train_val_test

plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["axes.grid"] = True


In [ ]:
method_name = "RBS"
val_count = 5000
test_count = 5000

repo_root = Path.cwd()
if not (repo_root / "artifacts").exists() and (repo_root.parent / "artifacts").exists():
    repo_root = repo_root.parent

artifact_dir = repo_root / "artifacts"
package_path = artifact_dir / f"lrn_{method_name.lower()}_package.zip"
x_scaled_path = artifact_dir / f"{method_name.lower()}_input_scaled.dat"
y_scaled_path = artifact_dir / f"{method_name.lower()}_energy_targets.dat"

if not package_path.exists():
    raise FileNotFoundError(f"Package not found: {package_path}")
if not x_scaled_path.exists():
    raise FileNotFoundError(f"Scaled input artifact not found: {x_scaled_path}")
if not y_scaled_path.exists():
    raise FileNotFoundError(f"Scaled target artifact not found: {y_scaled_path}")

with zipfile.ZipFile(package_path, "r") as archive:
    manifest = json.loads(archive.read("package.json").decode("utf-8"))
    onnx_name = manifest["model"]["onnx_file"]
    output_transform_info = manifest["preprocessing"]["output_transform"]
    temp_dir = Path(tempfile.mkdtemp(prefix="ibamlkit_onnx_"))
    archive.extract(onnx_name, path=temp_dir)

onnx_path = temp_dir / onnx_name
input_dim = int(manifest["model"]["input_dimension"])
output_dim = int(manifest["model"]["output_dimension"])
if output_dim <= 0:
    spectra_lengths = manifest["model"]["schema"]["outputs"]["spectra_lengths"]
    output_dim = int(sum(int(length) for length in spectra_lengths.values()))
sample_count = x_scaled_path.stat().st_size // (np.dtype(np.float32).itemsize * input_dim)

x_scaled = np.memmap(x_scaled_path, mode="r", dtype=np.float32, shape=(sample_count, input_dim))
y_scaled = np.memmap(y_scaled_path, mode="r", dtype=np.float32, shape=(sample_count, output_dim))

split = split_train_val_test(
    x_scaled,
    y_scaled,
    val_count=val_count,
    test_count=test_count,
)

x_test_scaled = split.test_inputs
y_test_scaled = split.test_targets

output_transform = ConstantFactorTransform(float(output_transform_info["factor"]))
output_transform.fit(np.zeros((1, output_dim), dtype=np.float32))

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Package:", package_path)
print("ONNX:", onnx_path)
print("Test inputs:", x_test_scaled.shape)
print("Test targets:", y_test_scaled.shape)


In [ ]:
plot_count = min(3, x_test_scaled.shape[0])
pred_scaled = session.run([output_name], {input_name: np.asarray(x_test_scaled[:plot_count], dtype=np.float32)})[0]

pred_e = np.empty_like(pred_scaled, dtype=np.float32)
y_e = np.empty_like(np.asarray(y_test_scaled[:plot_count], dtype=np.float32))
output_transform.inverse_transform(pred_scaled, out=pred_e)
output_transform.inverse_transform(y_test_scaled[:plot_count], out=y_e)

chi2 = np.mean((pred_e - y_e) ** 2 / (y_e + 1.0), axis=1)
print("Mean chi2:", float(np.mean(chi2)))
print("Median chi2:", float(np.median(chi2)))

fig, axes = plt.subplots(plot_count, 1, sharex=True)
if plot_count == 1:
    axes = [axes]
for row_index in range(plot_count):
    axes[row_index].plot(y_e[row_index], label="target")
    axes[row_index].plot(pred_e[row_index], label="prediction", linestyle="--")
    axes[row_index].set_title(f"Sample {row_index}")
    axes[row_index].legend()
fig.suptitle("Package test: E-space target vs prediction")
fig.tight_layout()


In [ ]:
import time

benchmark_batch = np.asarray(x_test_scaled, dtype=np.float32)
warmup_runs = 1
timed_runs = 3

for _ in range(warmup_runs):
    session.run([output_name], {input_name: benchmark_batch})

elapsed = []
for _ in range(timed_runs):
    start = time.perf_counter()
    session.run([output_name], {input_name: benchmark_batch})
    elapsed.append(time.perf_counter() - start)

elapsed = np.asarray(elapsed, dtype=np.float64)
samples_per_run = benchmark_batch.shape[0]
print(f"Benchmark batch: {benchmark_batch.shape}")
print(f"Runs: {timed_runs}, warmup: {warmup_runs}")
print(f"Mean latency: {elapsed.mean() * 1000:.2f} ms/run")
print(f"Median latency: {np.median(elapsed) * 1000:.2f} ms/run")
print(f"Throughput: {samples_per_run / elapsed.mean():.2f} samples/s")
